# 🇮🇳 Constitution of India — NLP Question Answering System

This notebook builds a **Retrieval-Augmented Generation (RAG)** pipeline that:
1. Downloads and parses the Constitution of India (PDF)
2. Splits it into searchable chunks
3. Creates semantic embeddings using `sentence-transformers`
4. Uses FAISS for fast similarity search
5. Answers user questions using a HuggingFace QA model

---
> **Runtime:** GPU recommended (Runtime → Change runtime type → T4 GPU)

## 📦 Step 1: Install Dependencies

In [ ]:
!pip install -q transformers sentence-transformers faiss-cpu pymupdf langchain langchain-community gradio

## 📥 Step 2: Download the Constitution of India PDF

In [ ]:
import urllib.request
import os

PDF_URL = "https://www.constitutinofofindia.net/constitution_of_india.pdf"
PDF_PATH = "constitution_of_india.pdf"

# --- Option A: Auto-download (try this first) ---
try:
    print("Attempting to download Constitution PDF...")
    urllib.request.urlretrieve(PDF_URL, PDF_PATH)
    print(f"✅ Downloaded to: {PDF_PATH}")
except Exception as e:
    print(f"⚠️  Auto-download failed: {e}")
    print()
    print("👉 MANUAL UPLOAD INSTRUCTIONS:")
    print("   1. Download the PDF from: https://legislative.gov.in/constitution-of-india/")
    print("   2. Click the 📁 Files icon in the left sidebar of Colab")
    print("   3. Upload the file and rename it to: constitution_of_india.pdf")
    print("   4. Re-run this cell after upload")

# Check if file exists
if os.path.exists(PDF_PATH):
    size_mb = os.path.getsize(PDF_PATH) / (1024 * 1024)
    print(f"📄 File ready: {PDF_PATH} ({size_mb:.1f} MB)")
else:
    print("❌ PDF not found. Please upload manually.")

## 📖 Step 3: Extract & Clean Text from PDF

In [ ]:
import fitz  # PyMuPDF
import re

def extract_text_from_pdf(pdf_path):
    """Extract and clean text from a PDF file."""
    doc = fitz.open(pdf_path)
    full_text = []

    print(f"📄 Total pages: {len(doc)}")

    for page_num, page in enumerate(doc):
        text = page.get_text("text")
        # Clean up whitespace and artifacts
        text = re.sub(r'\n{3,}', '\n\n', text)   # Collapse excessive newlines
        text = re.sub(r' {2,}', ' ', text)         # Collapse extra spaces
        text = text.strip()
        if text:
            full_text.append(text)

    combined = "\n\n".join(full_text)
    doc.close()
    return combined

raw_text = extract_text_from_pdf(PDF_PATH)
print(f"✅ Extracted {len(raw_text):,} characters of text")
print("\n--- Preview (first 800 chars) ---")
print(raw_text[:800])

## ✂️ Step 4: Split Text into Chunks

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

def split_into_chunks(text, chunk_size=600, chunk_overlap=100):
    """
    Split text into overlapping chunks.
    - chunk_size: max characters per chunk
    - chunk_overlap: overlap between chunks for context continuity
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    chunks = splitter.split_text(text)
    return chunks

chunks = split_into_chunks(raw_text)
print(f"✅ Created {len(chunks)} text chunks")
print(f"   Avg chunk length: {sum(len(c) for c in chunks) // len(chunks)} chars")
print("\n--- Sample chunk ---")
print(chunks[10])

## 🧠 Step 5: Create Semantic Embeddings + FAISS Index

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load embedding model
print("Loading embedding model...")
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Encode all chunks
print(f"Encoding {len(chunks)} chunks... (this may take a minute)")
embeddings = embed_model.encode(
    chunks,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

# Build FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype(np.float32))

print(f"\n✅ FAISS index built with {index.ntotal} vectors (dim={dimension})")

## 🔍 Step 6: Retrieval Function

In [ ]:
def retrieve_relevant_chunks(query, top_k=5):
    """
    Given a query, return the top_k most relevant text chunks.
    """
    query_vec = embed_model.encode([query], convert_to_numpy=True).astype(np.float32)
    distances, indices = index.search(query_vec, top_k)
    results = [chunks[i] for i in indices[0] if i < len(chunks)]
    return results

# Quick test
test_results = retrieve_relevant_chunks("Right to equality")
print("🔍 Test retrieval for 'Right to equality':")
for i, r in enumerate(test_results[:2], 1):
    print(f"\n[Chunk {i}]\n{r[:300]}...")

## 🤖 Step 7: Load QA Model for Answer Generation

In [ ]:
from transformers import pipeline

print("Loading QA model (deepset/roberta-base-squad2)...")
qa_pipeline = pipeline(
    "question-answering",
    model="deepset/roberta-base-squad2",
    tokenizer="deepset/roberta-base-squad2"
)
print("✅ QA model loaded!")

## 💬 Step 8: Full Q&A Pipeline

In [ ]:
def answer_question(question, top_k=5, score_threshold=0.05):
    """
    Full RAG pipeline:
    1. Retrieve relevant chunks using semantic search
    2. Run QA model over each chunk
    3. Return best answer with source context
    """
    if not question.strip():
        return "Please enter a valid question.", ""

    # Step 1: Retrieve
    relevant_chunks = retrieve_relevant_chunks(question, top_k=top_k)

    if not relevant_chunks:
        return "No relevant content found.", ""

    # Step 2: Run QA on each chunk, collect answers
    candidates = []
    for chunk in relevant_chunks:
        try:
            result = qa_pipeline(
                question=question,
                context=chunk,
                max_answer_len=200
            )
            if result["score"] >= score_threshold:
                candidates.append({
                    "answer": result["answer"],
                    "score": result["score"],
                    "context": chunk
                })
        except Exception:
            continue

    if not candidates:
        return "I could not find a confident answer. Try rephrasing your question.", relevant_chunks[0]

    # Step 3: Return best answer
    best = max(candidates, key=lambda x: x["score"])
    confidence = f"{best['score']:.0%}"
    return best["answer"], best["context"], confidence

# ---- Test it out ----
test_questions = [
    "What is the age requirement to become the President of India?",
    "What does Article 21 say?",
    "How many members are in the Rajya Sabha?",
]

for q in test_questions:
    result = answer_question(q)
    answer, context, conf = result if len(result) == 3 else (*result, "N/A")
    print(f"\n❓ Q: {q}")
    print(f"✅ A: {answer}")
    print(f"📊 Confidence: {conf}")
    print("-" * 60)

## 🖥️ Step 9: Interactive Gradio Chat UI

In [ ]:
import gradio as gr

def gradio_answer(question):
    result = answer_question(question)
    if len(result) == 3:
        answer, context, confidence = result
    else:
        answer, context = result
        confidence = "N/A"

    formatted_context = f"**Source Context:**\n> {context[:600]}..."
    return answer, confidence, formatted_context

with gr.Blocks(theme=gr.themes.Soft(), title="Constitution of India QA") as demo:
    gr.Markdown("""
    # 🇮🇳 Constitution of India — Q&A System
    Ask any question about the **Constitution of India**. The system will retrieve relevant
    clauses and extract the most accurate answer.
    """)

    with gr.Row():
        with gr.Column(scale=2):
            question_box = gr.Textbox(
                label="Your Question",
                placeholder="e.g. What are the Fundamental Rights of Indian citizens?",
                lines=3
            )
            submit_btn = gr.Button("🔍 Get Answer", variant="primary")

        with gr.Column(scale=3):
            answer_box = gr.Textbox(label="📋 Answer", lines=3, interactive=False)
            confidence_box = gr.Textbox(label="📊 Model Confidence", interactive=False)
            context_box = gr.Markdown(label="📖 Source Context")

    gr.Examples(
        examples=[
            "What are the fundamental rights of Indian citizens?",
            "What is the term of office of the President?",
            "What does Article 32 deal with?",
            "How is the Prime Minister of India appointed?",
            "What is the procedure for amendment of the Constitution?",
            "What is the official language of the Union?",
            "What are Directive Principles of State Policy?",
        ],
        inputs=question_box
    )

    submit_btn.click(
        fn=gradio_answer,
        inputs=question_box,
        outputs=[answer_box, confidence_box, context_box]
    )

demo.launch(share=True, debug=False)

## 🧪 Step 10: CLI Mode (Alternative — No UI)

In [ ]:
# Run this cell for a simple terminal-based loop (no Gradio needed)
print("🇮🇳 Constitution of India Q&A — Type 'exit' to quit\n")
print("=" * 60)

while True:
    user_input = input("\n❓ Your question: ").strip()
    if user_input.lower() in ["exit", "quit", "q"]:
        print("Goodbye! 🙏")
        break
    if not user_input:
        continue

    result = answer_question(user_input)
    if len(result) == 3:
        answer, context, confidence = result
        print(f"\n✅ Answer: {answer}")
        print(f"📊 Confidence: {confidence}")
        print(f"\n📖 Source (excerpt):\n   {context[:400]}...")
    else:
        answer, context = result
        print(f"\n⚠️  {answer}")
    print("-" * 60)

---
## 📚 Architecture Summary

| Component | Tool Used | Purpose |
|---|---|---|
| PDF Parsing | `PyMuPDF (fitz)` | Extract raw text from Constitution PDF |
| Text Chunking | `LangChain RecursiveCharacterTextSplitter` | Split into overlapping context windows |
| Embeddings | `all-MiniLM-L6-v2` (SentenceTransformers) | Dense semantic vectors for each chunk |
| Vector Search | `FAISS (IndexFlatL2)` | Fast nearest-neighbor retrieval |
| QA Model | `deepset/roberta-base-squad2` | Extractive answer from retrieved context |
| UI | `Gradio` | Browser-based interactive interface |

## 🔧 Tips & Customization
- **Better answers**: Try `deepset/deberta-v3-base-squad2` for a stronger QA model
- **Chunk size**: Reduce `chunk_size` to 400 for more precise answers; increase to 800 for more context
- **Top-K**: Increase `top_k` in `answer_question()` to search more chunks (slower but more thorough)
- **Multilingual**: Use `paraphrase-multilingual-MiniLM-L12-v2` embedding model for Hindi/regional queries